In [2]:
import xarray as xr
import pandas as pd
import os
import time
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from geopy.geocoders import Nominatim
import pandas as pd


# serial code start here

## load data serially

In [3]:
years = range(2012, 2022)

When we started working on the serial code, we got a lot of errors and we decided to use this script to check the coordinates names 

In [4]:
for year in years:
    file_path = f"./All Data 2012 - 2021/CHIRPS_total_precipitation_day_0.25x0.25_africa_{year}_v2.0.nc"
    ds = xr.open_dataset(file_path)
    print(year, list(ds.coords), list(ds.data_vars))
    ds.close()

2012 ['time', 'lon', 'lat'] ['pr']
2013 ['time', 'lon', 'lat'] ['pr']
2014 ['time', 'lon', 'lat'] ['pr']
2015 ['time', 'lon', 'lat'] ['pr']
2016 ['time', 'lon', 'lat'] ['pr']
2017 ['time', 'lon', 'lat'] ['pr']
2018 ['time', 'lon', 'lat'] ['pr']
2019 ['time', 'lon', 'lat'] ['pr']
2020 ['time', 'lon', 'lat'] ['pr']
2021 ['time', 'longitude', 'latitude'] ['pr']


From this we realised that,the errorst was due to the difference in terms of the location coordinates.
    From this we make sure to convert all the coordinates to the same nomination during the data loading before further actions 

In [ ]:
#data from 2021 use 'longitude', 'latitude' while data from the other years use 'lon', 'lat'

In [ ]:
data_list = [] # to store DataFrames from each year
t0 = time.perf_counter()
# Loop through each year, load the dataset, convert to DataFrame, and append to data_list
for year in years:
    print(f"Started processing file: CHIRPS_total_precipitation_day_0.25x0.25_africa_{year}_v2.0.nc")
    file_path = f"./All Data 2012 - 2021/CHIRPS_total_precipitation_day_0.25x0.25_africa_{year}_v2.0.nc"
    
    ds = xr.open_dataset(file_path)
    
    # Normalize coordinate names to lat/lon
    rename_map = {}
    if 'latitude' in ds.coords: rename_map['latitude'] = 'lat'
    if 'longitude' in ds.coords: rename_map['longitude'] = 'lon'
    if rename_map:
        ds = ds.rename(rename_map)
    
    df = ds.to_dataframe().dropna().reset_index()
    
    # Add year column
    df['time'] = pd.to_datetime(df['time'])
    df['year'] = df['time'].dt.year
    
    data_list.append(df)  # ← removed .sample(), append full DataFrame
    
    ds.close()
    print(f"finished processing file: CHIRPS_total_precipitation_day_0.25x0.25_africa_{year}_v2.0.nc")

data= pd.concat(data_list, ignore_index=True)
print(f"Time taken by serial code data loading: {time.perf_counter() - t0:.2f} seconds")


Started processing file: CHIRPS_total_precipitation_day_0.25x0.25_africa_2012_v2.0.nc
finished processing file: CHIRPS_total_precipitation_day_0.25x0.25_africa_2012_v2.0.nc
Started processing file: CHIRPS_total_precipitation_day_0.25x0.25_africa_2013_v2.0.nc
finished processing file: CHIRPS_total_precipitation_day_0.25x0.25_africa_2013_v2.0.nc
Started processing file: CHIRPS_total_precipitation_day_0.25x0.25_africa_2014_v2.0.nc
finished processing file: CHIRPS_total_precipitation_day_0.25x0.25_africa_2014_v2.0.nc
Started processing file: CHIRPS_total_precipitation_day_0.25x0.25_africa_2015_v2.0.nc
finished processing file: CHIRPS_total_precipitation_day_0.25x0.25_africa_2015_v2.0.nc
Started processing file: CHIRPS_total_precipitation_day_0.25x0.25_africa_2016_v2.0.nc
finished processing file: CHIRPS_total_precipitation_day_0.25x0.25_africa_2016_v2.0.nc
Started processing file: CHIRPS_total_precipitation_day_0.25x0.25_africa_2017_v2.0.nc
finished processing file: CHIRPS_total_precipitat

In [ ]:
print(len(data)) # size of the combined DataFrame

191340011


In [ ]:
data.head() # check the first 5 rows of the combined DataFrame

,time,lat,lon,pr,year
0,2012-01-01 12:00:00,-34.875,19.875,0.0,2012
1,2012-01-01 12:00:00,-34.625,19.375,0.0,2012
2,2012-01-01 12:00:00,-34.625,19.625,0.0,2012
3,2012-01-01 12:00:00,-34.625,19.875,0.0,2012
4,2012-01-01 12:00:00,-34.625,20.125,0.0,2012


In [ ]:
data.tail() # check the last 5 rows of the combined DataFrame

,time,lat,lon,pr,year
191340006,2021-08-31,49.875,143.125,3.473567,2021
191340007,2021-08-31,49.875,143.375,1.356130,2021
191340008,2021-08-31,49.875,143.625,0.326433,2021
191340009,2021-08-31,49.875,143.875,2.136942,2021
191340010,2021-08-31,49.875,144.125,6.475855,2021


## Statistic computation

### Compute average, and maximum precipitations per (lat, lon) across all years 

In [ ]:

agregation_per_location  = data.groupby(['lat', 'lon'])['pr'].agg(
    avg_precipitation  ='mean',
    max_precipitation  ='max'
).reset_index()
# round the results to 3 decimal places for better readability
agregation_per_location = agregation_per_location.round(3)

print(agregation_per_location.head())

      lat     lon  avg_precipitation  max_precipitation
0 -49.875 -75.375              7.505         100.190002
1 -49.875 -75.125              5.989          84.349998
2 -49.875 -74.875              4.254          95.897003
3 -49.875 -74.625              2.757          34.778000
4 -49.875 -74.375              2.285          32.830002


### use names to display data, implemented for 10 rows because it was too heavy for the laptop to compute all

In [ ]:
# Initialize geocoder
geolocator = Nominatim(user_agent="test")

# Create empty column if it doesn't exist
agregation_per_location['location'] = ""

# Loop through rows
for i in range(10):   # only first 10 rows
    lat = agregation_per_location.loc[i, 'lat']
    lon = agregation_per_location.loc[i, 'lon']
    
    try:
        location = geolocator.reverse(f"{float(lat)}, {float(lon)}")
        agregation_per_location.loc[i, 'location'] = location.address if location else "Not found"
    except Exception as e:
        agregation_per_location.loc[i, 'location'] = f"Error: {e}"
    
    time.sleep(1)  # add delay to avoid hitting geocoding service limits

print(agregation_per_location.head(10))

      lat     lon  avg_precipitation  max_precipitation  \
0 -49.875 -75.375              7.505         100.190002   
1 -49.875 -75.125              5.989          84.349998   
2 -49.875 -74.875              4.254          95.897003   
3 -49.875 -74.625              2.757          34.778000   
4 -49.875 -74.375              2.285          32.830002   
5 -49.875 -74.125              2.166          28.340000   
6 -49.875 -73.875              3.095          78.952003   
7 -49.875 -73.625              6.681         178.854996   
8 -49.875 -73.375              5.567          62.990002   
9 -49.875 -73.125              2.520          25.868000   

                                            location  
0  Natales, Provincia de Última Esperanza, Región...  
1  Natales, Provincia de Última Esperanza, Región...  
2  Natales, Huertos Endesa, Puerto Natales, Provi...  
3  Natales, Huertos Endesa, Puerto Natales, Provi...  
4  Natales, Huertos Endesa, Puerto Natales, Provi...  
5  Natales, Provinci

during our researches we found that Using GeoPy in serial code is preferable because geocoding is I/O-bound, not CPU-bound.
 It sends requests to online services like OpenStreetMap, so running many requests in parallel can cause rate limits, timeouts, or bans, and the overhead of parallel processing may even reduce performance.
 Serial execution is slower but more stable and API-friendly. 
 ##### from this we decided to not implement GeoPy it on the parallel code

###	Temporal Trend: Compute yearly average, and maximum precipitations and plot trend with linear regression.

In [18]:
# Group by year and calculate average and max precipitation
yearly_stats = data.groupby('year')['pr'].agg(
    avg_pr='mean',
    max_pr='max'
).reset_index()

# Linear regression on yearly average
slope, intercept, r, p, se = stats.linregress(yearly_stats['year'], yearly_stats['avg_pr'])
slope_max, intercept_max, r_max, p_max, se_max = stats.linregress(yearly_stats['year'], yearly_stats['max_pr'])
yearly_stats['trend'] = intercept + slope * yearly_stats['year']
yearly_stats['trendMax'] = intercept_max + slope_max * yearly_stats['year']

print(yearly_stats)
print(f"Mean Slope: {slope:.4f} mm/year | Mean R²: {r**2:.4f} | Mean p-value: {p:.4f}")
print(f"Max precip Slope: {slope_max:.4f} mm/year | Max precip R²: {r_max**2:.4f} | Max precip p-value: {p_max:.4f}")

   year    avg_pr       max_pr     trend    trendMax
0  2012  1.658738   366.000000  1.464095  301.661861
1  2013  1.594302   476.000000  1.519638  342.013663
2  2014  1.592540   453.000000  1.575180  382.365465
3  2015  1.511377   295.000000  1.630722  422.717267
4  2016  1.575540   321.000000  1.686265  463.069070
5  2017  1.632312   550.000000  1.741807  503.420872
6  2018  1.732177   367.000000  1.797349  543.772674
7  2019  1.725150   590.000000  1.852892  584.124476
8  2020  1.596485   288.000000  1.908434  624.476278
9  2021  2.521736  1126.449707  1.963976  664.828081
Mean Slope: 0.0555 mm/year | Mean R²: 0.3324 | Mean p-value: 0.0811
Max precip Slope: 40.3518 mm/year | Max precip R²: 0.2409 | Max precip p-value: 0.1498


### Extreme Events: Compute 95th percentile and count events where pr exceeds this threshold.

In [ ]:

threshold_95 = data['pr'].quantile(0.95)
extreme_events = data[data['pr'] > threshold_95]

print(f"95th percentile threshold : {threshold_95:.4f} mm")
print(f"Total extreme events      : {len(extreme_events):,}")
print(f"% of total records        : {100 * len(extreme_events) / len(data):.2f}%")

# Extreme events per year
extreme_per_year = extreme_events.groupby('year').size().reset_index(name='extreme_count')
print(extreme_per_year)

95th percentile threshold : 12.0000 mm
Total extreme events      : 8,752,315
% of total records        : 4.57%
   year  extreme_count
0  2012         745711
1  2013         695580
2  2014         685142
3  2015         656746
4  2016         702002
5  2017         729956
6  2018         758645
7  2019         752041
8  2020         343075
9  2021        2683417


### Plots

In [31]:

# ==============================
# Create output folder
# ==============================
output_dir = "./plots/serial"

years = yearly_stats['year'].values

# ==============================
# Plot 1: Yearly average + regression trend
# ==============================
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(years, yearly_stats['avg_pr'],
         marker='o', color='#378ADD',
         linewidth=2, markersize=6,
         label='Yearly average')

ax1.plot(years, yearly_stats['trend'],
         linestyle='--', color='#D85A30',
         linewidth=2,
         label=f'Trend (slope={slope:.4f} mm/yr, R²={r**2:.3f})')

ax1.fill_between(years,
                 yearly_stats['avg_pr'],
                 yearly_stats['trend'],
                 alpha=0.08, color='#378ADD')

ax1.set_title('Yearly average precipitation with linear regression', fontsize=13)
ax1.set_ylabel('Avg precipitation (mm)')
ax1.set_xlabel('Year')
ax1.set_xticks(years)
ax1.legend(fontsize=10)
ax1.grid(axis='y', linestyle='--', alpha=0.4)

plt.savefig(os.path.join(output_dir, 'plot1_yearly_avg_regression.png'),
            dpi=150, bbox_inches='tight')
plt.close()

# ==============================
# Plot 2: Yearly average + regression trend
# ==============================
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(years, yearly_stats['max_pr'],
         marker='o', color='#378ADD',
         linewidth=2, markersize=6,
         label='Yearly average')

ax1.plot(years, yearly_stats['trendMax'],
         linestyle='--', color='#D85A30',
         linewidth=2,
         label=f'Trend (slope={slope:.4f} mm/yr, R²={r**2:.3f})')

ax1.fill_between(years,
                 yearly_stats['max_pr'],
                 yearly_stats['trend'],
                 alpha=0.08, color='#378ADD')

ax1.set_title('Max average precipitation with linear regression', fontsize=13)
ax1.set_ylabel('Max precipitation (mm)')
ax1.set_xlabel('Year')
ax1.set_xticks(years)
ax1.legend(fontsize=10)
ax1.grid(axis='y', linestyle='--', alpha=0.4)

plt.savefig(os.path.join(output_dir, 'plot_yearly_max_regression.png'),
            dpi=150, bbox_inches='tight')
plt.close()
# ==============================
# Plot 3: Yearly maximum
# ==============================
fig, ax2 = plt.subplots(figsize=(10, 5))

bars = ax2.bar(years,
               yearly_stats['max_pr'],
               color='#5DCAA5',
               edgecolor='#0F6E56',
               linewidth=0.8,
               width=0.6)

for bar, val in zip(bars, yearly_stats['max_pr']):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.5,
             f'{val:.1f}',
             ha='center', va='bottom', fontsize=9)

ax2.set_title('Yearly maximum precipitation', fontsize=13)
ax2.set_ylabel('Max precipitation (mm)')
ax2.set_xlabel('Year')
ax2.set_xticks(years)
ax2.grid(axis='y', linestyle='--', alpha=0.4)

plt.savefig(os.path.join(output_dir, 'plot2_yearly_maximum.png'),
            dpi=150, bbox_inches='tight')
plt.close()

# ==============================
# Plot 4: Extreme event counts per year
# ==============================
fig, ax3 = plt.subplots(figsize=(10, 5))

bars3 = ax3.bar(extreme_per_year['year'],
                extreme_per_year['extreme_count'],
                color='#EF9F27',
                edgecolor='#BA7517',
                linewidth=0.8,
                width=0.6)

for bar, val in zip(bars3, extreme_per_year['extreme_count']):
    ax3.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 20,
             f'{val:,}',
             ha='center', va='bottom', fontsize=9)

ax3.axhline(extreme_per_year['extreme_count'].mean(),
            color='#A32D2D',
            linestyle='--',
            linewidth=1.5,
            label=f"Mean: {extreme_per_year['extreme_count'].mean():,.0f}")

ax3.set_title(
    f'Extreme events per year (threshold: {threshold_95:.2f} mm — 95th percentile)',
    fontsize=13
)
ax3.set_ylabel('Event count')
ax3.set_xlabel('Year')
ax3.set_xticks(extreme_per_year['year'])
ax3.legend(fontsize=10)
ax3.grid(axis='y', linestyle='--', alpha=0.4)

plt.savefig(os.path.join(output_dir, 'plot3_extreme_events_per_year.png'),
            dpi=400, bbox_inches='tight')
plt.close()

# ==============================
# Done
# ==============================
print(f"All plots saved successfully in: {output_dir}")

All plots saved successfully in: ./plots/serial


### End of serial code

We tried to do the parallele code here but when we started working on it using multiprocessing it was not working. After some researches we discovered on stackoverflow that several people encountered the same issue because of the usage of Jupyter Notebook (https://stackoverflow.com/questions/47313732/jupyter-notebook-never-finishes-processing-using-multiprocessing-python-3). We tried the solutions proposed but it was not working for us, so we decided to exit from jupyter notebook, write the code on Visual Studio Code and use the same conda environment on the terminal to execute the code and this was working.

We attempted to implement the project using Dask to see compare it to the multiprocessing in order to select the best one. However, we encountered several issues during implementation. While the dynamic (automatic) core allocation in Dask worked correctly, fixing the computation to a specific number of cores (e.g., 5 cores , selected after doing a performance evaluation too) led to execution errors and instability in the pipeline.

Since the objective of the study was to identify an optimal number of cores and consistently apply it across the full workflow to allow the replication of the result, this limitation prevented us from using the controlled Dask. Therefore, we decided to retain the multiprocessing implementation, which provided stable and reproducible results for core scalability analysis.

You could get assess to all our code in this github repository: https://github.com/ahmadkongodev/hpc_project_wascal

Thank you for reading our notebook.


##### by APPAH Esther  (eappah@jm.uesd.edu.gh),     KONGO Hamado (ahmadkongo5@gmail.com),   SANYANG Binta (binTeddy1989@gmail.com) in April 2026